In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    import faiss
    print("faiss ok")
except Exception as e:
    print("faiss import failed:", e)


faiss ok


In [2]:
CAST_ROOT = Path('/home/toploc1/Datasets/conversational/CAST2019')
INDEX_ROOT = Path('/home/toploc1/Datasets/toploc1/indexes')

print('CAST_ROOT exists:', CAST_ROOT.exists())
print('INDEX_ROOT exists:', INDEX_ROOT.exists())


CAST_ROOT exists: True
INDEX_ROOT exists: True


In [ ]:
def list_files(root: Path, n=200):
    items = []
    for p in sorted(root.rglob('*')):
        if p.is_file():
            items.append((str(p), p.suffix, p.stat().st_size))
    df = pd.DataFrame(items, columns=['path', 'suffix', 'size_bytes'])
    display(df.head(n))
    print('num files:', len(df))
    return df

cast_files = list_files(CAST_ROOT)
index_files = list_files(INDEX_ROOT)


In [ ]:
keywords = ['parquet', 'index', 'faiss', 'npy', 'qrel', 'query', 'cast2019', 'cast2020', 'snowflake', 'dragon', 'hnsw']

def filter_paths(df, terms):
    mask = pd.Series(False, index=df.index)
    for t in terms:
        mask = mask | df['path'].str.contains(t, case=False, regex=False)
    return df[mask].copy()

display(filter_paths(cast_files, keywords).head(300))
display(filter_paths(index_files, keywords).head(300))


In [7]:
# Example placeholders — edit these
DOC_EMB_DIR = CAST_ROOT / 'snowflake_embeddings' 
TRAIN_QUERY_DIR = CAST_ROOT / 'snowflake_embeddings' 
DEV_QUERY_DIR = CAST_ROOT /  'snowflake_embeddings'
HNSW_INDEX_PATH = INDEX_ROOT / 'treccast_hnsw.index'
DOC_IDS_PATH = INDEX_ROOT / 'treccast_hnsw_ids.npy'

print(DOC_EMB_DIR)
print(TRAIN_QUERY_DIR)
print(DEV_QUERY_DIR)
print(HNSW_INDEX_PATH)
print(DOC_IDS_PATH)


/home/toploc1/Datasets/conversational/CAST2019/snowflake_embeddings
/home/toploc1/Datasets/conversational/CAST2019/snowflake_embeddings
/home/toploc1/Datasets/conversational/CAST2019/snowflake_embeddings
/home/toploc1/Datasets/toploc1/indexes/treccast_hnsw.index
/home/toploc1/Datasets/toploc1/indexes/treccast_hnsw_ids.npy


In [8]:
def parquet_files(folder: Path):
    return sorted(folder.glob('*.parquet'))

def inspect_one_parquet(folder: Path):
    files = parquet_files(folder)
    if not files:
        print('No parquet files in', folder)
        return None
    f = files[0]
    df = pd.read_parquet(f)
    print('file:', f)
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    display(df.head(3))
    return df

doc_sample = inspect_one_parquet(DOC_EMB_DIR)
train_q_sample = inspect_one_parquet(TRAIN_QUERY_DIR)
dev_q_sample = inspect_one_parquet(DEV_QUERY_DIR)


file: /home/toploc1/Datasets/conversational/CAST2019/snowflake_embeddings/cast2019_snowflake_v2.rank0.part00000.parquet
shape: (200000, 2)
columns: ['id', 'embedding']


,id,embedding
0,MARCO_0,"[0.04559326, 0.056762695, 0.019454956, -0.0003..."
1,MARCO_1,"[0.010047913, 0.0602417, 0.070739746, 0.007904..."
2,MARCO_2,"[0.026229858, 0.06378174, 0.06524658, -0.01835..."


file: /home/toploc1/Datasets/conversational/CAST2019/snowflake_embeddings/cast2019_snowflake_v2.rank0.part00000.parquet
shape: (200000, 2)
columns: ['id', 'embedding']


,id,embedding
0,MARCO_0,"[0.04559326, 0.056762695, 0.019454956, -0.0003..."
1,MARCO_1,"[0.010047913, 0.0602417, 0.070739746, 0.007904..."
2,MARCO_2,"[0.026229858, 0.06378174, 0.06524658, -0.01835..."


file: /home/toploc1/Datasets/conversational/CAST2019/snowflake_embeddings/cast2019_snowflake_v2.rank0.part00000.parquet
shape: (200000, 2)
columns: ['id', 'embedding']


,id,embedding
0,MARCO_0,"[0.04559326, 0.056762695, 0.019454956, -0.0003..."
1,MARCO_1,"[0.010047913, 0.0602417, 0.070739746, 0.007904..."
2,MARCO_2,"[0.026229858, 0.06378174, 0.06524658, -0.01835..."


In [9]:
ID_COL = 'id'
EMB_COL = 'embedding'

def load_small_embeddings(folder: Path, n_files=1, n_rows=1000):
    all_ids = []
    all_vecs = []
    for f in parquet_files(folder)[:n_files]:
        df = pd.read_parquet(f, columns=[ID_COL, EMB_COL]).head(n_rows)
        all_ids.extend(df[ID_COL].astype(str).tolist())
        all_vecs.append(np.array(df[EMB_COL].tolist(), dtype=np.float32))
    X = np.vstack(all_vecs) if all_vecs else np.empty((0, 0), dtype=np.float32)
    return np.array(all_ids, dtype=object), X

doc_ids_small, doc_emb_small = load_small_embeddings(DOC_EMB_DIR)
train_ids_small, train_emb_small = load_small_embeddings(TRAIN_QUERY_DIR)
dev_ids_small, dev_emb_small = load_small_embeddings(DEV_QUERY_DIR)

print('doc:', doc_emb_small.shape)
print('train query:', train_emb_small.shape)
print('dev query:', dev_emb_small.shape)


doc: (1000, 1024)
train query: (1000, 1024)
dev query: (1000, 1024)


In [10]:
def norm_stats(x, name):
    if len(x) == 0:
        print(name, 'empty')
        return
    norms = np.linalg.norm(x, axis=1)
    print(name)
    print('  min :', float(norms.min()))
    print('  mean:', float(norms.mean()))
    print('  max :', float(norms.max()))
    print('  std :', float(norms.std()))

norm_stats(doc_emb_small, 'doc_emb_small')
norm_stats(train_emb_small, 'train_emb_small')
norm_stats(dev_emb_small, 'dev_emb_small')


doc_emb_small
  min : 0.9996790289878845
  mean: 0.9999974370002747
  max : 1.0003286600112915
  std : 0.0001763658510753885
train_emb_small
  min : 0.9996790289878845
  mean: 0.9999974370002747
  max : 1.0003286600112915
  std : 0.0001763658510753885
dev_emb_small
  min : 0.9996790289878845
  mean: 0.9999974370002747
  max : 1.0003286600112915
  std : 0.0001763658510753885


In [ ]:
if HNSW_INDEX_PATH.exists():
    idx = faiss.read_index(str(HNSW_INDEX_PATH))
    print('index ntotal:', idx.ntotal)
else:
    print('Index file not found')

if DOC_IDS_PATH.exists():
    saved_doc_ids = np.load(DOC_IDS_PATH, allow_pickle=True)
    print('saved doc ids shape:', saved_doc_ids.shape)
    print('first 5 ids:', saved_doc_ids[:5])
else:
    print('Doc ids file not found')
